# Day 083 — Exercise 2: Dependency Ordering (Topological Sort)

**What you'll build:** `topo_sort` — given a list of tasks with dependency ids, return them in an order where every dependency comes before the task that needs it.

**Why it matters:** the LLM may return tasks in any order. Executing them naively would try to 'Write summary' before 'Gather facts'. Topological sort is the classic algorithm for scheduling tasks with dependencies, and it's the only thing standing between you and executing a plan backwards.

In [ ]:
import json

_PLAN_JSON = json.dumps([
    {'id': 't1', 'title': 'Gather facts',
     'description': 'Collect the relevant information.', 'depends_on': []},
    {'id': 't2', 'title': 'Draft outline',
     'description': 'Organize the facts into an outline.', 'depends_on': ['t1']},
    {'id': 't3', 'title': 'Write summary',
     'description': 'Write the final summary.', 'depends_on': ['t2']},
])

def _mock_planner(plan_json=None, task_result='Task done.'):
    """Return an llm_fn: the plan JSON on planning calls, task_result on execution calls."""
    plan = plan_json if plan_json is not None else _PLAN_JSON
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'json array' in system.lower() or 'planning' in system.lower():
            return plan
        return task_result
    return _fn

def _mock_executor(task):
    return 'Result: ' + task.title
import json
from dataclasses import dataclass, field

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, list) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the Task dataclass ────────────────────────────────────────────────────────
@dataclass
class Task:
    """One step in a plan.

    Attributes:
        id:          short snake_case identifier (e.g. 't1', 'write_outline').
        title:       brief human-readable label (5 words max).
        description: one sentence describing what to do.
        depends_on:  ids of tasks that must complete before this one.
        status:      'pending' | 'done' | 'failed'.
        result:      the output of executing this task.
    """
    id: str
    title: str
    description: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    result: str = ""

# ── planning: ask the LLM to decompose a goal ─────────────────────────────────
def build_plan_prompt(goal, context=None):
    """Build a prompt that asks the LLM to break a goal into a JSON task list."""
    system = "\n".join([
        "You are a planning assistant. Break the goal into an ordered list of tasks.",
        "",
        "Return ONLY a JSON array. Each item must have these exact keys:",
        '  "id": short snake_case identifier (t1, t2, ...)',
        '  "title": brief label (5 words max)',
        '  "description": one sentence - what to do',
        '  "depends_on": list of task ids that must finish before this one ([] if none)',
        "",
        "Return ONLY the JSON array. No prose, no markdown fences.",
    ])
    user_parts = ["Goal: " + str(goal)]
    if context:
        user_parts.append("Context: " + str(context))
    return [{"role": "system", "content": system},
            {"role": "user", "content": "\n".join(user_parts)}]


def parse_plan(text):
    """Extract a task list from LLM output. Returns list[Task]; never raises.

    Tolerates markdown fences, prose before/after, missing fields, and invalid
    JSON. Invalid or missing fields are filled with safe defaults so any
    parseable item becomes a valid Task.
    """
    items = safe_parse_list(text) or []
    tasks = []
    for i, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        tasks.append(Task(
            id=str(item.get("id", "t" + str(i + 1))),
            title=str(item.get("title", "Task " + str(i + 1))),
            description=str(item.get("description", "")),
            depends_on=[str(d) for d in item.get("depends_on", [])
                        if isinstance(d, str)],
        ))
    return tasks


## Task

`topo_sort(tasks) -> list[Task]` using Kahn's algorithm:

1. Build `in_deg = {task.id: 0}` for each task; for each task's `depends_on`, increment `in_deg[task.id]` for each dep that exists.
2. Seed a queue with all tasks where `in_deg == 0`.
3. While the queue has tasks: pop, emit, then decrement `in_deg` for every task that depended on the one just emitted; enqueue any that hit zero.
4. Any tasks not emitted (cycle victims) are appended at the end. **Never raises.**

## Your Implementation

In [ ]:
def topo_sort(tasks):
    """Sort tasks so every dependency runs before the task that needs it.
    Uses Kahn's algorithm; appends cyclic tasks at the end rather than raising.
    """
    raise NotImplementedError


In [ ]:

# ── dependency ordering: Kahn's topological sort ──────────────────────────────
def topo_sort(tasks):
    """Sort tasks so every dependency comes before the task that needs it.

    Uses Kahn's algorithm (BFS on a DAG). If a cycle exists the cyclic tasks
    are appended at the end in their original order rather than raising, so
    execution can still proceed on the non-cyclic portion.
    """
    by_id = {t.id: t for t in tasks}
    # count incoming edges (how many unresolved deps each task has)
    in_deg = {t.id: 0 for t in tasks}
    for t in tasks:
        for dep in t.depends_on:
            if dep in in_deg:
                in_deg[t.id] += 1
    # start with tasks that have no deps
    queue = [t.id for t in tasks if in_deg[t.id] == 0]
    order = []
    while queue:
        tid = queue.pop(0)
        order.append(by_id[tid])
        # for every task that listed tid as a dep, reduce its in-degree
        for t in tasks:
            if tid in t.depends_on:
                in_deg[t.id] -= 1
                if in_deg[t.id] == 0:
                    queue.append(t.id)
    # cycle guard: any task not yet emitted has a circular dependency
    done_ids = {t.id for t in order}
    for t in tasks:
        if t.id not in done_ids:
            order.append(t)
    return order


## Automated checks

In [ ]:

score, total = 0, 5
try:
    tasks = parse_plan(_PLAN_JSON)     # t1 -> t2 -> t3 chain
    ordered = topo_sort(tasks)
    ids = [t.id for t in ordered]
    assert ids.index('t1') < ids.index('t2') < ids.index('t3')
    score += 1; print("✅ linear chain is sorted t1 -> t2 -> t3")

    no_deps = [Task('a','A','a'), Task('b','B','b'), Task('c','C','c')]
    assert len(topo_sort(no_deps)) == 3
    score += 1; print("✅ tasks with no dependencies are all included")

    # diamond: d1, d2 depend on base; final depends on both
    diamond = [
        Task('base','Base','do'), Task('d1','D1','do', depends_on=['base']),
        Task('d2','D2','do', depends_on=['base']),
        Task('final','Final','do', depends_on=['d1','d2']),
    ]
    di = topo_sort(diamond)
    di_ids = [t.id for t in di]
    assert di_ids.index('base') < di_ids.index('d1')
    assert di_ids.index('base') < di_ids.index('d2')
    assert di_ids.index('d1') < di_ids.index('final')
    assert di_ids.index('d2') < di_ids.index('final')
    score += 1; print("✅ diamond dependency resolved correctly")

    cycle = [Task('x','X','x',depends_on=['y']), Task('y','Y','y',depends_on=['x'])]
    result = topo_sort(cycle)
    assert len(result) == 2   # cycle: both tasks still returned, not dropped
    score += 1; print("✅ cycle detected: tasks still returned (not dropped or raised)")

    assert topo_sort([]) == []
    score += 1; print("✅ empty task list returns empty list")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── dependency ordering: Kahn's topological sort ──────────────────────────────
def topo_sort(tasks):
    """Sort tasks so every dependency comes before the task that needs it.

    Uses Kahn's algorithm (BFS on a DAG). If a cycle exists the cyclic tasks
    are appended at the end in their original order rather than raising, so
    execution can still proceed on the non-cyclic portion.
    """
    by_id = {t.id: t for t in tasks}
    # count incoming edges (how many unresolved deps each task has)
    in_deg = {t.id: 0 for t in tasks}
    for t in tasks:
        for dep in t.depends_on:
            if dep in in_deg:
                in_deg[t.id] += 1
    # start with tasks that have no deps
    queue = [t.id for t in tasks if in_deg[t.id] == 0]
    order = []
    while queue:
        tid = queue.pop(0)
        order.append(by_id[tid])
        # for every task that listed tid as a dep, reduce its in-degree
        for t in tasks:
            if tid in t.depends_on:
                in_deg[t.id] -= 1
                if in_deg[t.id] == 0:
                    queue.append(t.id)
    # cycle guard: any task not yet emitted has a circular dependency
    done_ids = {t.id for t in order}
    for t in tasks:
        if t.id not in done_ids:
            order.append(t)
    return order
```

**Why append cycle victims instead of raising?** Raising on a circular dependency stops the whole plan. Appending them means the non-cyclic portion of the plan can still execute, and the cyclic tasks at least run in some order rather than being silently dropped.

</details>